# ACE-Step 1.5 Remote GPU Server

Run this notebook on **Colab with GPU** (Runtime → Change runtime type → T4 GPU).

After the last cell runs it prints an ngrok URL — copy that and set it as `ACE_STEP_ENDPOINT` in your local `.env` or environment before starting the FastAPI server.

In [ ]:
# Cell 1 — detect platform + verify GPU
import os, subprocess

# Auto-detect Colab vs Kaggle
if os.path.exists('/kaggle'):
    BASE_DIR = '/kaggle/working'
    print('Platform: Kaggle')
else:
    BASE_DIR = '/content'
    print('Platform: Colab')

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout.strip() or 'No GPU found — switch runtime to GPU before continuing')

In [ ]:
# Cell 2 — install dependencies
!pip install -q fastapi uvicorn[standard] pyngrok requests
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q 'transformers>=4.40' diffusers accelerate soundfile scipy

# Install ACE-Step from source
!git clone -q https://github.com/ace-step/ACE-Step.git /content/ACE-Step
!pip install -q -e /content/ACE-Step

import sys
sys.path.insert(0, '/content/ACE-Step')
print('Dependencies installed')

In [ ]:
# Cell 3 — HuggingFace login + download model weights
HF_TOKEN = ''   # <-- paste your HuggingFace read-token here

from huggingface_hub import login, snapshot_download
import os

if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN above — the model requires authentication')

login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK')

MODEL_ID  = 'ACE-Step/Ace-Step1.5'
LOCAL_DIR = f'{BASE_DIR}/ace_step_model'

os.makedirs(LOCAL_DIR, exist_ok=True)
print(f'Downloading {MODEL_ID} → {LOCAL_DIR} (~12 GB)...')
snapshot_download(repo_id=MODEL_ID, local_dir=LOCAL_DIR,
                  local_dir_use_symlinks=False, token=HF_TOKEN)
print('Model download complete')

In [ ]:
# Cell 4 — load pipeline into VRAM
import os, torch

# T4 (SM 7.5) doesn't support Flash Attention 2 — force eager/sdpa attention
os.environ["ATTN_IMPLEMENTATION"] = "eager"
os.environ["XFORMERS_DISABLED"] = "1"

from acestep.pipeline_ace_step import ACEStepPipeline

if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available — switch the Colab runtime to GPU')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB')

pipeline = ACEStepPipeline(
    checkpoint_dir=LOCAL_DIR,
    dtype='bfloat16',
    device='cuda',
    cpu_offload=vram_gb < 20,
    quantized=vram_gb < 6,
)
print('ACE-Step pipeline loaded and ready')

In [ ]:
# Cell 5 — FastAPI server + ngrok tunnel
# Safe to re-run: kills any existing server on port 8765 first.
import subprocess, time
subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)
time.sleep(1)

import uuid, threading, traceback
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel
import uvicorn, torch

app = FastAPI()

_jobs: dict[str, dict] = {}
_RESULTS_DIR = Path(f'{BASE_DIR}/results')
_RESULTS_DIR.mkdir(exist_ok=True)


class GenerateRequest(BaseModel):
    audio_duration: int = 30
    prompt: str = ''
    lyrics: str = ''
    infer_step: int = 60
    guidance_scale: float = 7.0
    scheduler_type: str = 'euler'
    guidance_interval: float = 0.5
    use_erg_tag: bool = True
    use_erg_lyric: bool = False
    use_erg_diffusion: bool = False


def _run_generation(job_id: str, req: GenerateRequest):
    out_path = _RESULTS_DIR / f'{job_id}.wav'
    try:
        _jobs[job_id]['status'] = 'processing'
        import inspect
        sig = inspect.signature(pipeline.__call__)
        params = sig.parameters

        kwargs = dict(
            audio_duration=req.audio_duration,
            prompt=req.prompt,
            lyrics=req.lyrics,
            infer_step=req.infer_step,
            guidance_scale=req.guidance_scale,
            scheduler_type=req.scheduler_type,
            save_path=str(out_path),
        )
        if 'guidance_interval' in params:
            kwargs['guidance_interval'] = req.guidance_interval
        if 'use_erg_tag' in params:
            kwargs['use_erg_tag'] = req.use_erg_tag
            kwargs['use_erg_lyric'] = req.use_erg_lyric
            kwargs['use_erg_diffusion'] = req.use_erg_diffusion

        with torch.inference_mode():
            pipeline(**kwargs)

        _jobs[job_id] = {'status': 'done', 'wav_path': str(out_path), 'error': None}
        print(f'[{job_id[:8]}] done')
    except Exception as exc:
        print(f'[{job_id[:8]}] FAILED:\n{traceback.format_exc()}')
        _jobs[job_id] = {'status': 'failed', 'wav_path': None, 'error': str(exc)}


@app.post('/generate')
def generate(req: GenerateRequest):
    job_id = str(uuid.uuid4())
    _jobs[job_id] = {'status': 'pending', 'wav_path': None, 'error': None}
    threading.Thread(target=_run_generation, args=(job_id, req), daemon=True).start()
    return {'job_id': job_id}


@app.get('/result/{job_id}')
def result(job_id: str):
    job = _jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail='Job not found')
    if job['status'] == 'done':
        return FileResponse(job['wav_path'], media_type='audio/wav', filename=f'{job_id}.wav')
    if job['status'] == 'failed':
        return JSONResponse({'status': 'failed', 'error': job['error']})
    return JSONResponse({'status': job['status']})


@app.get('/health')
def health():
    return {'status': 'ok', 'gpu': torch.cuda.get_device_name(0)}


def _start_server():
    uvicorn.run(app, host='0.0.0.0', port=8765, log_level='warning')

threading.Thread(target=_start_server, daemon=True).start()
time.sleep(2)

from pyngrok import ngrok

NGROK_AUTHTOKEN = ''   # <-- paste your ngrok authtoken

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
time.sleep(1)
tunnel = ngrok.connect(8765)
public_url = tunnel.public_url

print('=' * 60)
print(f'ACE-Step GPU server is live!')
print(f'Public URL: {public_url}')
print(f'\nUpdate your .env:  ACE_STEP_ENDPOINT={public_url}')
print('=' * 60)

In [ ]:
# Cell 6 — Keep-alive (prevents Kaggle/Colab idle timeout)
# Run this after Cell 5. It pings the pipeline every 10 min to stay active.
import threading, time

def _keepalive():
    while True:
        time.sleep(600)
        try:
            _ = pipeline          # prevent idle GPU timeout
            _ = list(_jobs)       # keep memory active
        except Exception:
            pass

threading.Thread(target=_keepalive, daemon=True).start()
print('Keep-alive running — session will stay active up to the 12h limit')